# Librerias

In [13]:
#Importing libraries
import numpy as np
import pandas as pd
from datetime import datetime, timedelta
pd.options.display.max_rows = 999
import warnings
warnings.filterwarnings("ignore")
import sys
sys.dont_write_bytecode = True
#importing datasets
from src.New_Utils import New_Sequences
from src.New_Utils import Sequences_Night
from Datasets import Replace
from Datasets import AIDE
from Datasets import Shanghai
#Libraries to run the experiments
from src.model import Model
from sklearn.model_selection import StratifiedKFold
skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
import os
import psutil
from joblib import Parallel, delayed

# Codigo General

## Training

In [ ]:
model = Model()
Data_ROC_train = []
Data_ROC_test = []
results_train = []
results_test = []

l1_penalty=0.01
l2_penalty=0.001
learning_rate=0.000001
batch_size=64
sequences = New_Sequences(Replace)

In [ ]:
# loop across the folds
for i, (train_index, test_index) in enumerate(skf.split(X=[s['X'] for s in sequences], y=[s['Y'] for s in sequences])):
    
        # obtaining the folds to use
    train_fold = [sequences[idx] for idx in train_index]
    test_fold = [sequences[idx] for idx in test_index]
    
    # fit the model to the training set
    model.fit(
        sequences=train_fold,
        sequence_length=int(7 * 24 * 60 // 5),
        l1_penalty=l1_penalty,
        l2_penalty=l2_penalty,
        learning_rate=learning_rate,
        batch_size=batch_size,
        epochs=1000,
        seed=42,
        verbose=0
    )
    # predicting the values on the fold's data
    train_probs = np.array(model.predict(sequences=train_fold)).flatten()
    test_probs = np.array(model.predict(sequences=test_fold)).flatten()
    # Create a new list for holding the sequences, the mark of the sequence and the predicted probability
    for idx, seq in enumerate(train_fold):
        new_seq = seq.copy()
        new_seq['Y_prob'] = train_probs[idx]
        new_seq['fold'] = i + 1
        Data_ROC_train.append(new_seq)

    # 4. Procesamos el Test
    for idx, seq in enumerate(test_fold):
        new_seq = seq.copy()
        new_seq['Y_prob'] = test_probs[idx]
        new_seq['fold'] = i + 1
        Data_ROC_test.append(new_seq)
        # evaluate the model on the traiing and testing sets
        metrics_train = model.evaluate(sequences=train_fold)
        metrics_test = model.evaluate(sequences=test_fold)
    # save the results
    results_train.append(metrics_train)
    results_test.append(metrics_test)
    

# Optimized

In [14]:

def process_fold(i, train_index, test_index, sequences, params):
    #initial RAM monitoring before paralellizing the CPU usage
    process = psutil.Process(os.getpid())
    mem_inicio = process.memory_info().rss / (1024 ** 2) #show in MB
    
    train_fold = [sequences[idx] for idx in train_index]
    test_fold = [sequences[idx] for idx in test_index]
    
    local_model = model
    
    local_model.fit(
        sequences=train_fold,

        sequence_length=params['seq_len'], #not sure this is real

        l1_penalty=params['l1'],
        l2_penalty=params['l2'],
        learning_rate=params['lr'],
        batch_size=params['bs'],
        epochs=1000,
        seed=42,
        verbose=0
    )
    
    train_probs = np.array(local_model.predict(sequences=train_fold)).flatten()
    test_probs = np.array(local_model.predict(sequences=test_fold)).flatten()
    
    m_train = local_model.evaluate(sequences=train_fold)
    m_test = local_model.evaluate(sequences=test_fold)
    
    # 5. Preparación de datos para ROC
    fold_train_results = []
    for idx, seq in enumerate(train_fold):
        new_seq = {
            'patient': seq['patient'],
            'Y': seq['Y'],
            'Y_prob': train_probs[idx],
            'fold': i + 1,
            'L': seq['L']
        }
        fold_train_results.append(new_seq)

    fold_test_results = []
    for idx, seq in enumerate(test_fold):
        new_seq = {
            'patient': seq['patient'],
            'Y': seq['Y'],
            'Y_prob': test_probs[idx],
            'fold': i + 1,
            'L': seq['L']
        }
        fold_test_results.append(new_seq)

    # --- RAM monitoring ---
    mem_final = process.memory_info().rss / (1024 ** 2)
    print(f"✅ Fold {i+1} completo | RAM usada: {mem_final - mem_inicio:.2f} MB | Total proceso: {mem_final:.2f} MB")
    
    return fold_train_results, fold_test_results, m_train, m_test

In [ ]:
# Running the above function
from joblib import Parallel, delayed, parallel_backend
N_JOBS = 8 
params_config = {
    'seq_len': int(7 * 24 * 60 // 5),
    'l1': l1_penalty, 
    'l2': l2_penalty,
    'lr': learning_rate, 
    'bs': batch_size
}

y_labels = [s['Y'] for s in sequences]

print(f"Iniciando CV con {N_JOBS} núcleos (Backend: multiprocessing)...")
# Paralellized execution


# El context manager 'parallel_backend' ayuda a evitar el conflicto de inicialización
with parallel_backend('multiprocessing'):
    resultados_paralelos = Parallel(n_jobs=N_JOBS)(
        delayed(process_fold)(i, t_idx, v_idx, sequences, params_config) 
        for i, (t_idx, v_idx) in enumerate(skf.split(X=np.zeros(len(y_labels)), y=y_labels))
    )

# Unifying results
for f_train, f_test, m_train, m_test in resultados_paralelos:
    Data_ROC_train.extend(f_train)
    Data_ROC_test.extend(f_test)
    results_train.append(m_train)
    results_test.append(m_test)

### Test

In [16]:
from src.model import Model
params = {
    'seq_len': int(7 * 24 * 60 // 5),
    'l1': l1_penalty, 
    'l2': l2_penalty,
    'lr': learning_rate, 
    'bs': batch_size
}

local_model = Model()
local_model.fit(
    sequences=train_fold,
    sequence_length=params['seq_len'],
    l1_penalty=params['l1'],
    l2_penalty=params['l2'],
    learning_rate=params['lr'],
    batch_size=params['bs'],
    epochs=1000,
    seed=42,
    verbose=0
)
metrics = local_model.evaluate(sequences=test_fold)

In [17]:
metrics

{'accuracy': 0.7159590043923866,
 'balanced_accuracy': 0.7237584650112867,
 'precision': 0.8373983739837398,
 'sensitivity': 0.6975169300225733,
 'specificity': 0.75,
 'f1': 0.7610837438423645,
 'auc': 0.7929928517682469}

## External Validation

### Replace

In [ ]:
sequences = New_Sequences(Replace)
Results_Replace = model.evaluate(sequences = sequences)

In [ ]:
# For External Validation (after training is done)
Replace_external_sequences = sequences.copy()
external_probs = model.predict(sequences=sequences)
for i in range(len(Replace_external_sequences)):
    Replace_external_sequences[i]['Y_prob'] =    external_probs[i]

### AIDE

In [ ]:
sequences = New_Sequences(AIDE)
Results_AIDE = model.evaluate(Sequences = sequences)

In [ ]:
# For External Validation (after training is done)
AIDE_external_sequences = sequences.copy()
external_probs = model.predict(sequences=sequences)
for i in range(len(AIDE_external_sequences)):
    AIDE_external_sequences[i]['Y_prob'] =    external_probs[i]

### Shanghai

## ROC

In [ ]:
import pandas as pd
from sklearn.metrics import roc_curve, auc
import matplotlib.pyplot as plt

df_test_results = pd.DataFrame(Data_ROC_test)

for fold_num in df_test_results['fold'].unique():
    fold_data = df_test_results[df_test_results['fold'] == fold_num]
    fpr, tpr, _ = roc_curve(fold_data['Y'], fold_data['Y_prob'])
    plt.plot(fpr, tpr, alpha=0.3, label=f'Fold {fold_num}')

# Codigo Noche

## Secuencias de noche

In [ ]:
sequences = Sequences_Night(Replace)
sequences_noche = [seq for seq in sequences if seq['Y_total'] == 1]

## Modelo noche

In [ ]:
model_night = Model()
results_train = []
results_test = []

Data_ROC_train = []
Data_ROC_test = []

l1_penalty=0.01
l2_penalty=0.01
learning_rate=0.000001
batch_size=16

In [ ]:
# loop across the folds
for i, (train_index, test_index) in enumerate(skf.split(X=[s['X'] for s in sequences], y=[s['Y'] for s in sequences])):
    
        # obtaining the folds to use
    train_fold = [sequences[idx] for idx in train_index]
    test_fold = [sequences[idx] for idx in test_index]
    
    # fit the model to the training set
    model_night.fit(
        sequences=train_fold,
        sequence_length=int(7 * 24 * 60 // 5),
        l1_penalty=l1_penalty,
        l2_penalty=l2_penalty,
        learning_rate=learning_rate,
        batch_size=batch_size,
        epochs=1000,
        seed=42,
        verbose=0
    )
    # predicting the values on the fold's data
    train_probs = np.array(model_night.predict(sequences=train_fold)).flatten()
    test_probs = np.array(model_night.predict(sequences=test_fold)).flatten()
    # Create a new list for holding the sequences, the mark of the sequence and the predicted probability
    for idx, seq in enumerate(train_fold):
        new_seq = seq.copy()
        new_seq['Y_prob'] = train_probs[idx]
        new_seq['fold'] = i + 1
        Data_ROC_train.append(new_seq)

    # 4. Procesamos el Test
    for idx, seq in enumerate(test_fold):
        new_seq = seq.copy()
        new_seq['Y_prob'] = test_probs[idx]
        new_seq['fold'] = i + 1
        Data_ROC_test.append(new_seq)
        # evaluate the model on the traiing and testing sets
        metrics_train = model_night.evaluate(sequences=train_fold)
        metrics_test = model_night.evaluate(sequences=test_fold)
    # save the results
    results_train.append(metrics_train)
    results_test.append(metrics_test)
    

## Validacion Externa

### AIDE

In [ ]:
sequences = New_Sequences(AIDE)
Results_AIDE = model.evaluate(Sequences = sequences)

In [ ]:
# For External Validation (after training is done)
AIDE_external_sequences = sequences.copy()
external_probs = model.predict(sequences=sequences)
for i in range(len(AIDE_external_sequences)):
    AIDE_external_sequences[i]['Y_prob'] =    external_probs[i]

## ROC

In [ ]:
import pandas as pd
from sklearn.metrics import roc_curve, auc
import matplotlib.pyplot as plt

df_test_results = pd.DataFrame(Data_ROC_test)

for fold_num in df_test_results['fold'].unique():
    fold_data = df_test_results[df_test_results['fold'] == fold_num]
    fpr, tpr, _ = roc_curve(fold_data['Y'], fold_data['Y_prob'])
    plt.plot(fpr, tpr, alpha=0.3, label=f'Fold {fold_num}')